# Milestone 2: RAG Exploration

This notebook has the initial explorations of the RAG pipeline.

In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
import faiss
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

/Users/yasamanbaher/miniforge3/envs/project-575/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
meta = pd.read_json("../data/raw/meta_Toys_and_Games.jsonl", lines=True, nrows=10000)
review = pd.read_json("../data/raw/Toys_and_Games.jsonl", lines=True, nrows=10000)

cleaned_meta = meta.drop(columns = ['videos', 'price', 'images', 'bought_together', 'subtitle', 'author'])
cleaned_meta.head()

reviews = review[review['verified_purchase'] == True]
cleaned_reviews = reviews.drop(columns = ['images', 'timestamp', 'user_id', 'verified_purchase'])
cleaned_reviews.head()

,rating,title,text,asin,parent_asin,helpful_vote
0,5,Granddaughters love them!,I purchased several of these for my granddaugh...,B09QH7QJS7,B09QH7QJS7,0
1,3,Arrived broken,"It’s cute, but it arrived broken & has some pr...",B06XYKSKQP,B06XYKSKQP,1
2,5,Dylan loves them!!!,Bough for my grandson Dylan. He loves them.,B07SFF3YQW,B07XRSD5R9,0
3,5,Janod quality and very cute...,This was great for my daughters circus themed ...,B007JWWUDW,B007JWWUDW,0
4,3,I used for my daughters circus birthday party ...,This is very cute but beware that the animals ...,B00MZG6OO8,B00MZG6OO8,2


In [39]:
cleaned_meta = meta.drop(columns=['videos', 'price', 'images', 'bought_together', 'subtitle', 'author'], errors='ignore')

reviews = review[review['verified_purchase'] == True]
cleaned_reviews = reviews.drop(columns=['images', 'timestamp', 'user_id', 'verified_purchase'], errors='ignore')

cleaned_meta['description'] = cleaned_meta['description'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else (x if isinstance(x, str) else "")
)

cleaned_meta['details'] = cleaned_meta['details'].apply(
    lambda x: " ".join([f"{k} {v}" for k, v in x.items()]) if isinstance(x, dict) else ""
)

cleaned_meta['features'] = cleaned_meta['features'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
)

cleaned_meta['categories'] = cleaned_meta['categories'].apply(
    lambda x: " ".join(x) if isinstance(x, list) else ""
)

cleaned_meta = cleaned_meta[
    (cleaned_meta['title'].fillna('').str.strip() != '') &
    (cleaned_meta['description'].fillna('').str.strip() != '') &
    (cleaned_meta['features'].fillna('').str.strip() != '') &
    (cleaned_meta['categories'].fillna('').str.strip() != '')
].reset_index(drop=True)

In [40]:
review_text_cols = [col for col in ['title', 'text'] if col in cleaned_reviews.columns]

cleaned_reviews['combined_review_text'] = cleaned_reviews[review_text_cols].fillna('').agg(' '.join, axis=1)

grouped_reviews = (
    cleaned_reviews.groupby('parent_asin')['combined_review_text']
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

rag_df = cleaned_meta.merge(grouped_reviews, on='parent_asin', how='left')
rag_df['combined_review_text'] = rag_df['combined_review_text'].fillna('')

In [41]:
products = (
    rag_df['title'] + ' ' +
    rag_df['description'] + ' ' +
    rag_df['features'] + ' ' +
    rag_df['categories'] + ' ' +
    rag_df['combined_review_text']
).tolist()

In [42]:
model = SentenceTransformer("all-MiniLM-L6-v2")
product_embeddings = model.encode(products).astype("float32")

index = faiss.IndexFlatL2(product_embeddings.shape[1])
index.add(product_embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7058.12it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [49]:
# -------------------------
# Retriever
# -------------------------
def retrieve(query, top_k=5):
    query_embedding = model.encode([query]).astype("float32")
    distances, indices = index.search(query_embedding, top_k)
    return rag_df.iloc[indices[0]]

# -------------------------
# Building context
# -------------------------
# Implemented with the help of chatGPT
def build_context(docs):
    blocks = []
    for _, row in docs.iterrows():
        review_snippet = row.get('combined_review_text', '')[:500]

        block = (
            f"Product ASIN: {row.get('parent_asin', 'N/A')}\n"
            f"Title: {row.get('title', '')}\n"
            f"Description: {row.get('description', '')}\n"
            f"Features: {row.get('features', '')}\n"
            f"Categories: {row.get('categories', '')}\n"
            f"Review Evidence: {review_snippet}\n"
        )
        blocks.append(block)
    return "\n\n".join(blocks)


# -------------------------
# Wrapper function
# -------------------------
def retrieve_and_build_context(query):
    docs = retrieve(query, top_k=5)
    return build_context(docs)

In [50]:
prompt1 = ChatPromptTemplate.from_template(

"""
You must answer using ONLY the information in the context.

- If the answer is present, extract and summarize it clearly.
- Do NOT say "I don't know" if the answer exists in the context.
- Only say "I don't know" if the context truly does not contain the answer.

Context:
{context}

Question:
{input}

Answer:
"""
)

prompt2 = ChatPromptTemplate.from_template(

"""
You must answer using ONLY the information in the context.

- Keep the answer shorter than 3 sentences.
- Make sure nothing is repeated, and only necessary details are written.
- If the answer is not in the context, say: "The context does not provide enough information."

Context:
{context}

Input:
{input}

Answer:
"""
)

prompt3 = ChatPromptTemplate.from_template(

"""
You must answer using ONLY the information in the context.

- Be clear and helpful
- Give specific statements instead of general ones. 
- If something is missing, say "not enough context to answer your question"

Context:
{context}

Question:
{input}

Answer:
"""
)

In [55]:
llm = ChatGroq(model="llama-3.1-8b-instant")

rag_chain = (
    {
        "context": RunnableLambda(retrieve_and_build_context),
        "input": RunnablePassthrough()
    }
    | prompt1
    | llm
    | StrOutputParser()
)

In [56]:
queries = [
    "A good board game for kids age 8 and up",
    "A toy for toddlers",
    "Educational toys for kids"
]

for q in queries:
    print(f"\nQUERY: {q}")
    print(rag_chain.invoke(q))


===== prompt1 =====
Considering the context, a good board game for kids age 8 and up that is mentioned is the Sorry! Board Game (Product ASIN: B00000IWD0). The description states that children under age 8 may find it hard to handle the frustration of losing pawns, but it's suitable for kids as young as 6. However, it's likely that most kids would enjoy it by the age of 8.

===== prompt2 =====
The Sorry! game is known to be hard for children under age 8 to handle due to the frustration caused by sending each other's pawns back to the starting line.

===== prompt3 =====
Based on the description of the Sorry! Board Game, it's recommended for kids ages 6 and up. However, it's mentioned that children under 8 may have a hard time handling the frustration of having their pawns sent back to the starting line.

If you're looking for a board game suitable for kids age 8 and up, the Chinese Checkers game (Product ASIN: B08K7ST2ZT) might be a good option, as it's recommended for ages 7 and up.


In [57]:
prompts = {
    "prompt1": prompt1,
    "prompt2": prompt2,
    "prompt3": prompt3
}

query = "A good board game for kids age 8 and up"

for name, prompt in prompts.items():
    test_chain = (
        {
            "context": RunnableLambda(retrieve_and_build_context),
            "input": RunnablePassthrough()
        }
        | prompt
        | llm
        | StrOutputParser()
    )

    print(f"\n===== {name} =====")
    print(test_chain.invoke(query))


===== prompt1 =====
Based on the context, the answer to a good board game for kids age 8 and up is the Sorry! game. 

The description states, "This classic game of luck, strategy, and determination is easy to grasp for children as young as 6 years old, yet it's fun for adults and older siblings too." However, it also mentions that "This kind of frustration may be hard for children under age 8 to handle" and that children under 8 "typically crumble into tears of outrage when their pawns are cavalierly sent back."

Therefore, considering the statement that children under 8 might find the game too frustrating, the Sorry! game is suitable for kids age 8 and up.

===== prompt2 =====
The context does not provide enough information about a specific board game suitable for an 8-year-old child.

===== prompt3 =====
Not enough context to answer your question, but the Sorry! Board Game description mentions that the game may be hard for children under age 8 to handle, and children under age 8 typ